In [ ]:
# Cell A: Setup
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(123)

In [ ]:
# Cell B: Generate raw sensor data
n_readings = 1000
sensors = ['sensor_A', 'sensor_B', 'sensor_C']

raw_data = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-01', periods=n_readings, freq='5min'),
    'sensor_id': np.random.choice(sensors, n_readings),
    'temperature': np.random.normal(25, 5, n_readings),
    'humidity': np.random.normal(60, 10, n_readings),
    'pressure': np.random.normal(1013, 5, n_readings)
})

In [ ]:
# Cell C: Add some anomalies/outliers
anomaly_indices = np.random.choice(len(raw_data), 50, replace=False)
raw_data.loc[anomaly_indices, 'temperature'] += np.random.choice([-20, 20], 50)
raw_data.loc[anomaly_indices[:25], 'humidity'] = np.random.uniform(0, 10, 25)

In [ ]:
# Cell D: Clean data - remove extreme outliers
cleaned_data = raw_data.copy()

# Remove temperature outliers (> 3 std from mean)
temp_mean = cleaned_data['temperature'].mean()
temp_std = cleaned_data['temperature'].std()
temp_mask = (cleaned_data['temperature'] > temp_mean - 3*temp_std) & \
            (cleaned_data['temperature'] < temp_mean + 3*temp_std)

cleaned_data = cleaned_data[temp_mask].copy()

In [ ]:
# Cell E: Add derived features
cleaned_data['hour'] = cleaned_data['timestamp'].dt.hour
cleaned_data['day_of_week'] = cleaned_data['timestamp'].dt.dayofweek
cleaned_data['is_business_hours'] = cleaned_data['hour'].between(9, 17)

In [ ]:
# Cell F: Compute rolling statistics per sensor
rolling_stats = []
for sensor in sensors:
    sensor_data = cleaned_data[cleaned_data['sensor_id'] == sensor].copy()
    sensor_data['temp_rolling_mean'] = sensor_data['temperature'].rolling(12, min_periods=1).mean()
    sensor_data['temp_rolling_std'] = sensor_data['temperature'].rolling(12, min_periods=1).std()
    rolling_stats.append(sensor_data)

data_with_rolling = pd.concat(rolling_stats, ignore_index=True)

In [ ]:
# Cell G: Aggregate by sensor and hour
hourly_stats = data_with_rolling.groupby(['sensor_id', 'hour']).agg({
    'temperature': ['mean', 'std', 'min', 'max'],
    'humidity': ['mean', 'std'],
    'pressure': 'mean'
}).round(2)

hourly_stats.columns = ['_'.join(col) for col in hourly_stats.columns]

In [ ]:
# Cell H: Compute sensor correlations
pivot_temp = cleaned_data.pivot_table(
    index='timestamp', 
    columns='sensor_id', 
    values='temperature'
).dropna()

sensor_correlations = pivot_temp.corr()

In [ ]:
# Cell I: Create summary report DataFrames
sensor_summary = cleaned_data.groupby('sensor_id').agg({
    'temperature': ['mean', 'std', 'min', 'max', 'count'],
    'humidity': ['mean', 'std'],
    'pressure': ['mean', 'std']
}).round(2)

sensor_summary.columns = ['_'.join(col) for col in sensor_summary.columns]

In [ ]:
# Cell J: Final pipeline output
pipeline_results = {
    'raw_records': len(raw_data),
    'cleaned_records': len(cleaned_data),
    'records_removed': len(raw_data) - len(cleaned_data),
    'unique_sensors': len(sensors),
    'date_range': (cleaned_data['timestamp'].min(), cleaned_data['timestamp'].max()),
    'avg_temp_all': cleaned_data['temperature'].mean(),
    'avg_humidity_all': cleaned_data['humidity'].mean()
}